### BASELINE - ALGORITMO GENÉTICO

#### IMPORTS

In [50]:
from baseline.POSSIBLE import DISTRICTS_POINTS as DP
import math
import random
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import shapely.geometry

##### DECLARAÇÃO DE TIPOS E VARIÁVEIS

In [51]:
POPULATION_NUMBER = 1
ITERATIONS = 1
DISASTER_COUNTDOWN_EVERY = 30
PERCENTAGE_OF_DEATHS = 0.5

NUMBER_LOCATIONS = len(DP)
NUMBER_AMBUS_TYPE_A = 1
NUMBER_AMBUS_TYPE_B = 4
LOCATIONS = [] 
DISTANCE_MATRIX = []
RAIO = 0.0084
#10min 15min, 30min, 1h
#0.0056 0.0084, 0.0168, 0.0336



##### MÉTODOS AUXILIARES

In [52]:
def random_number(start, end):
    return random.randrange(start, end)

#### ENTIDADE LOCATION

In [53]:
# Região com lat e long e densidade populacional (peso)
class Location:
    def __init__(self, *args):
        self.id = args[0]
        self.coord_x = args[1]
        self.coord_y = args[2]
        self.weight = args[3]

##### FUNÇÕES REFERENTES A DISTANCIA E LOCALIZAÇÃO

In [54]:
def calc_distance(x1, y1, x2, y2):
    return math.dist((x1, y1), (x2, y2))
    
def prepare_locations():
    keys = list(DP)
    for i in range(NUMBER_LOCATIONS):
        local = Location(keys[i], DP[keys[i]][0], DP[keys[i][1]], DP[keys[i]][2])
        LOCATIONS.append(local)

def create_distance_matrix():
    for i in range(NUMBER_LOCATIONS):
        distances = []
        for j in range(NUMBER_LOCATIONS):
            distances.append(calc_distance(LOCATIONS[i].coord_x, LOCATIONS[i].coord_y, LOCATIONS[j].coord_x, LOCATIONS[j].coord_y))
        DISTANCE_MATRIX.append(distances)

#### ENTIDADE SOLUTION

In [55]:
class Solution:
    def __init__(self):
        self.xA = [0] * len(DP)
        self.xB = [0] * len(DP)
        self.non_zeros_A = []
        self.non_zeros_B = []
        self.rank = 0
        self.coverage_points = []

    def __lt__(self, other):
        return self.rank < other.rank
    
    def __gt__(self, other):
        return self.rank > other.rank

#### ALGORITMO GENÉTICO

In [ ]:
def create_solution():
    new_solution = Solution()
    for _ in range(NUMBER_AMBUS_TYPE_A):
        rando_index = random_number(0, NUMBER_LOCATIONS)
        if(rando_index not in new_solution.non_zeros_A):
            new_solution.non_zeros_A.append(rando_index)
            new_solution.xA[rando_index] = 1
            print(f'INDICE - AMBULANCIA DO TIPO A: {rando_index}')
    
    for _ in range(NUMBER_AMBUS_TYPE_B):
        rando_index = random_number(0, NUMBER_LOCATIONS)
        if(rando_index not in new_solution.non_zeros_B):
            new_solution.non_zeros_B.append(rando_index)
            new_solution.xB[rando_index] = 1
            print(f'INDICE - AMBULANCIA DO TIPO B: {rando_index}')
            
    return new_solution
    

def evaluate_solution(solution: Solution):
    solution.coverage_points = [0]*NUMBER_LOCATIONS

    for j in range(NUMBER_LOCATIONS):
        for i in range(len(solution.non_zeros_A)):
            if((DISTANCE_MATRIX[solution.non_zeros_A[i]][j] <= RAIO)):
                print(f"{DISTANCE_MATRIX[solution.non_zeros_A[i]][j]}")
                if(solution.xA[solution.non_zeros_A[i]] == 1):
                    solution.coverage_points[j] = 1
        for i in range(len(solution.non_zeros_B)):
            if(DISTANCE_MATRIX[solution.non_zeros_B[i]][j] <= RAIO):
                print(f"{DISTANCE_MATRIX[solution.non_zeros_B[i]][j]}")
                if(solution.xB[solution.non_zeros_B[i]] == 1):
                    solution.coverage_points[j] = 1

    for i in range(len(solution.coverage_points)):
        if(LOCATIONS[i].weight != "nan"):
            if(solution.coverage_points[i] == 1):
                solution.rank += float(LOCATIONS[i].weight)
        

def crossover_auxiliar():
    new_value = random_number(0,1)
    if(new_value == 0):
        return 0
    else:
        return 1
    

def crossover(base: Solution, guia:Solution):
    nova_solucao = Solution()

    for i in range(NUMBER_LOCATIONS):
        if(base.xA[i] == guia.xA[i]):
            nova_solucao.xA[i] = (base.xA[i])
        elif(base.xA[i] == 1 and guia.xA[i] == 0):
            nova_solucao.xA[i] = base.xA[i]
        else:
            nova_solucao.xA[i] = crossover_auxiliar()

        if(nova_solucao.xA[i] == 1):
            nova_solucao.non_zeros_A.append(i)
    
        if(base.xB[i] == guia.xB[i]):
            nova_solucao.xB[i] = (base.xB[i])
        elif(base.xB[i] == 1 and guia.xB[i] == 0):
            nova_solucao.xB[i] = base.xB[i]
        else:
            nova_solucao.xB[i] = crossover_auxiliar()

        if(nova_solucao.xB[i] == 1):
            nova_solucao.non_zeros_B.append(i)        
        
    evaluate_solution(nova_solucao)
    return nova_solucao


def choose_parent(population: list[Solution], type_parent):
    temp_pop = []
    sum_rank = 0

    for i in range(POPULATION_NUMBER):
        sum_rank += population[i].rank
    
    for i in range(POPULATION_NUMBER):
        temp_pop.append((population[i].rank)/(sum_rank))

    rank_benchmark = random.uniform(0,1)
    if(type_parent == 1):
        for i in range(POPULATION_NUMBER):
            if rank_benchmark < temp_pop[i]:
                return population[i]
    else:
        for i in range(POPULATION_NUMBER):
            if rank_benchmark > temp_pop[i]:
                return population[i]
    
    j = random_number(0, POPULATION_NUMBER - 1)
    return population[j]
